# 05 · Backtesting & the Lookahead Trap
**Goal:** build a tiny backtest of a moving-average strategy, then *see* how a one-line 'lookahead' bug (peeking at the future) manufactures fake profits — and how trading costs eat into the honest version.

> Maps to: project **08** (backtester). KB §08 and §1.7 (the golden rules).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(3)

## 1. A price series and an SMA-crossover signal
A **Simple Moving Average (SMA)** is just the average price over the last N days. A classic rule: be **long** (hold) when the short SMA is above the long SMA, otherwise be **flat** (out).

In [ ]:
n = 252 * 3
price = pd.Series(100 * np.cumprod(1 + rng.normal(0.0003, 0.012, n)))
sma_fast = price.rolling(10).mean()
sma_slow = price.rolling(50).mean()
signal = (sma_fast > sma_slow).astype(int)   # 1 = want to be long, 0 = flat
mkt_ret = price.pct_change()

## 2. The lookahead trap
Here's the crucial bit. The signal at the **close** of day *t* is computed from day *t*'s price. You can only **act on it the next day**. So your position for day *t* must use *yesterday's* signal:

- ✅ **Honest:** `position = signal.shift(1)` (act on the next bar)
- ❌ **Lookahead bug:** `position = signal` (trade on a price you couldn't know yet)

In [ ]:
honest_pos   = signal.shift(1)
lookahead_pos = signal            # the bug

honest_ret    = (honest_pos   * mkt_ret).fillna(0)
lookahead_ret = (lookahead_pos * mkt_ret).fillna(0)

eq_honest    = (1 + honest_ret).cumprod()
eq_lookahead = (1 + lookahead_ret).cumprod()
eq_bh        = (1 + mkt_ret.fillna(0)).cumprod()

plt.figure(figsize=(9,3.5))
eq_lookahead.plot(label='LOOKAHEAD bug (fake!)')
eq_honest.plot(label='honest (next-bar)')
eq_bh.plot(label='buy & hold', ls='--', color='grey')
plt.legend(); plt.title('The same strategy, with and without peeking'); plt.ylabel('equity'); plt.show()

The lookahead curve looks fantastic — and it's a **lie**. It 'knew' each day's move before trading. This is the #1 way backtests fool people.

## 3. Costs make it honest
Every time you switch position you pay: **commission** + **slippage**. Let's charge a few **basis points** (1 bps = 0.01%) per change in position and see the honest strategy's real picture.

In [ ]:
def sharpe(r):
    return np.sqrt(252) * r.mean() / r.std() if r.std() > 0 else 0.0

cost_bps = 5
turnover = honest_pos.diff().abs().fillna(0)
honest_ret_net = honest_ret - turnover * cost_bps / 1e4

print(f'Sharpe  lookahead (fake):   {sharpe(lookahead_ret):+.2f}')
print(f'Sharpe  honest (gross):     {sharpe(honest_ret):+.2f}')
print(f'Sharpe  honest (after cost):{sharpe(honest_ret_net):+.2f}')

## 4. More examples: how costs scale with trading frequency
The more often you trade, the more costs bite. Sweep the cost level and also a faster signal to see turnover matter.

In [ ]:
for cost_bps in [0, 2, 5, 10, 20]:
    net = honest_ret - turnover * cost_bps/1e4
    print(f'cost {cost_bps:2d} bps ->  net Sharpe {sharpe(net):+.2f}   total cost drag {turnover.sum()*cost_bps/1e4:.2%}')

print(f'\n(this strategy changes position {int(turnover.sum())} times over {len(price)} days)')

A strategy that looks fine at 0 bps can become a guaranteed loser at realistic costs — *especially* fast/high-turnover ones. Always test net of costs.

### 🧪 Try it yourself
1. Speed up the signal (`rolling(5)` vs `rolling(20)`): more crossings → more turnover → costs hurt more.
2. Add **slippage** as an extra fixed give-up per trade and fold it into `cost_bps`.
3. Allow short-selling: use `signal*2 - 1` (so −1 = short) instead of `signal` and see how it changes the equity curve.

**You should see:** the lookahead Sharpe is absurdly high; the honest gross Sharpe is modest or negative; after costs it's worse still. A naive crossover has no real edge — which is exactly what project **08** demonstrates with its PSR flag.

### In the project
Project **08** makes lookahead *structurally impossible* by stepping bar-by-bar through an event loop (`backtester/engine.py`) and charges commission + slippage by default (`backtester/execution.py`).